In [ ]:
import os
import pickle
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_recall_curve, average_precision_score, 
    brier_score_loss, auc
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

print("=========================================")
print("  E-COMMERCE MODEL COMPARISON & CALIBRATION")
print("=========================================\n")

# Ensure artifacts directory exists
os.makedirs('../../backend/artifacts', exist_ok=True)

# 1. Load the Data
train_df = pd.read_csv('../../data/processed/train.csv')
test_df = pd.read_csv('../../data/processed/test.csv')

X_train = train_df.drop('Revenue', axis=1)
y_train = train_df['Revenue']
X_test = test_df.drop('Revenue', axis=1)
y_test = test_df['Revenue']

# 2. Load the Trained Models for Comparison
print("Loading trained models for comparison...\n")
models = {}
model_files = [
    'logistic_regression.pkl', 
    'decision_tree.pkl', 
    'random_forest.pkl', 
    'xgboost.pkl'
]

for filename in model_files:
    filepath = f"../../backend/models/{filename}"
    if os.path.exists(filepath):
        with open(filepath, 'rb') as f:
            # Format name for display (e.g., 'xgboost.pkl' -> 'XGBoost')
            display_name = filename.replace('.pkl', '').replace('_', ' ').title()
            if display_name == 'Xgboost': display_name = 'XGBoost'
            models[display_name] = pickle.load(f)

# 3. Precision-Recall Curve (PR-AUC)
# Crucial for imbalanced datasets (84/16 split)
fig, ax = plt.subplots(figsize=(10, 8))
print("--- Calculating Precision-Recall AUC ---")

for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    
    print(f"{name} PR-AUC: {pr_auc:.4f}")
    ax.plot(recall, precision, lw=2, label=f"{name} (PR-AUC = {pr_auc:.3f})")

# Baseline represents the random guessing rate
baseline = y_test.mean()
ax.plot([0, 1], [baseline, baseline], color='navy', lw=2, linestyle='--', label=f'Baseline ({baseline:.3f})')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("Recall (True Positive Rate)")
ax.set_ylabel("Precision (Positive Predictive Value)")
ax.set_title("Precision-Recall Curve Comparison")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../../backend/artifacts/pr_curve_comparison.png')
plt.close()
print("Saved PR Curve -> artifacts/pr_curve_comparison.png\n")

# 4. Reliability Diagram (Calibration Curve)
# Shows how well predicted probabilities match actual outcomes
fig = plt.figure(figsize=(10, 8))
ax1 = plt.subplot2grid((3, 1), (0, 0), rowspan=2)
ax2 = plt.subplot2grid((3, 1), (2, 0))

ax1.plot([0, 1], [0, 1], "k:", label="Perfectly Calibrated")
print("--- Calculating Brier Scores (Lower is Better) ---")

for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    
    brier = brier_score_loss(y_test, y_prob)
    print(f"{name} Brier Score: {brier:.4f}")
    
    fraction_of_positives, mean_predicted_value = calibration_curve(y_test, y_prob, n_bins=10)
    
    ax1.plot(mean_predicted_value, fraction_of_positives, "s-", label=f"{name} (Brier = {brier:.3f})")
    ax2.hist(y_prob, range=(0, 1), bins=10, label=name, histtype="step", lw=2)

ax1.set_ylabel("Fraction of Positives (Actual)")
ax1.set_title("Calibration Plots (Reliability Diagrams)")
ax1.legend(loc="upper left")
ax1.grid(alpha=0.3)

ax2.set_xlabel("Mean Predicted Value (Probability)")
ax2.set_ylabel("Count")
ax2.legend(loc="upper center", ncol=2)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../../backend/artifacts/calibration_curve_uncalibrated.png')
plt.close()
print("Saved Uncalibrated Reliability Diagram -> artifacts/calibration_curve_uncalibrated.png\n")

# 5. Probability Calibration (Isotonic Regression)
print("--- Calibrating Best Model for True Customer Intent Probabilities ---")

# Load the dynamically chosen best model
with open('../../backend/models/best_model.pkl', 'rb') as f:
    best_model = pickle.load(f)

# Split X_test to avoid leaking data during calibration evaluation
X_calib, X_holdout, y_calib, y_holdout = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, stratify=y_test
)

# Record the uncalibrated Brier Score on the holdout set
uncalibrated_prob = best_model.predict_proba(X_holdout)[:, 1]
uncalibrated_brier = brier_score_loss(y_holdout, uncalibrated_prob)

# Use cv="prefit" to save compute time (model is already trained)
calibrated_model = CalibratedClassifierCV(estimator=best_model, method='isotonic', cv='prefit')
calibrated_model.fit(X_calib, y_calib)

# Test the newly calibrated model on the holdout set
calibrated_prob = calibrated_model.predict_proba(X_holdout)[:, 1]
calibrated_brier = brier_score_loss(y_holdout, calibrated_prob)

print(f"Uncalibrated Brier Score: {uncalibrated_brier:.4f}")
print(f"Calibrated Brier Score:   {calibrated_brier:.4f}")

if calibrated_brier < uncalibrated_brier:
    print("(Calibration successfully reduced probability error.)\n")
else:
    print("(Calibration did not significantly improve probabilities on this split, but is preserved for structural consistency.)\n")

# 6. Save the Calibrated Model for Production
# Save as a NEW file to ensure the notebook remains idempotent (can be rerun safely)
calibrated_model_path = '../../backend/models/calibrated_best_model.pkl'

with open(calibrated_model_path, 'wb') as f:
    pickle.dump(calibrated_model, f)
    
print(f"Saved Calibrated Model safely to -> {calibrated_model_path}")
print("(Use this file in the FastAPI backend for accurate percentage estimates)")

print("\n=========================================")
print("MODEL COMPARISON & CALIBRATION COMPLETE")
print("=========================================")